# Constrained and Guided Generation Tutorial

## Overview

This tutorial explores constrained and guided generation using a **locally-hosted open-source LLM** (Qwen3 14B-Instruct) via HuggingFace Transformers, and the **`outlines`** library for type-safe structured output.

## Motivation

LLMs sometimes produce outputs that are too open-ended or lack specific formats. Constrained generation techniques let us control the model's outputs more precisely. Traditionally this was done with post-hoc parsing (e.g., regex on free-form text), but libraries like **`outlines`** can enforce structure *during* generation — the model is mathematically prevented from producing tokens that violate the schema.

## Key Components

1. **Prompt-based constraints** — instructing the model via the prompt to follow a format
2. **Native regex parsing** — post-hoc extraction of structured fields from free-form output
3. **`outlines` structured generation** — guaranteed valid JSON/schema output via constrained decoding
4. **`outlines` classification** — using `Literal` types to restrict output to a fixed set of choices

## Method Details

We progress through three levels of control:

1. **Prompt engineering only** — we ask the model to follow a format. Simple but brittle; the model may deviate.
2. **Regex post-processing** — we generate free-form text, then extract fields with regex. Works, but fails silently if the model doesn't match the pattern.
3. **`outlines` constrained decoding** — we define a Pydantic model or `Literal` type, and `outlines` masks the token logits at each step so that *only* valid continuations are sampled. The output is **guaranteed** to be valid JSON conforming to the schema.

> **Note:** `outlines` works with any HuggingFace causal LM — no API calls or special model support required.

## Conclusion

By the end of this tutorial, you'll understand the progression from soft (prompt-based) to hard (schema-enforced) constraints and know when to reach for each approach.

## Setup

We install dependencies, load a 4-bit quantized Qwen3 14B-Instruct model, and prepare both a native `generate()` helper and an `outlines` structured model.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch outlines

import re
import torch
from typing import Literal
from pydantic import BaseModel
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import outlines

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Native generate helper (for unconstrained generation)
def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = hf_tokenizer([text], return_tensors="pt").to(hf_model.device)
    output_ids = hf_model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return hf_tokenizer.decode(generated, skip_special_tokens=True)

# Outlines structured model (for constrained generation)
structured_model = outlines.from_transformers(hf_model, hf_tokenizer)

## The Logit Pipeline: How Token Generation Actually Works

Before we explore constrained generation, we need to understand **exactly** how a language model turns a prompt into a single next token. Every generated token follows a five-stage pipeline:

### Stage 1: Tokenization → Input IDs
The input text is split into **tokens** (subword units) and converted to integer IDs. Each ID maps to a row in the model’s embedding table.

### Stage 2: Transformer Forward Pass → Logits
The token embeddings flow through the transformer’s layers (self-attention + feed-forward networks), producing **hidden states** at each position. The hidden state at the *last* position is projected through a **linear layer** (the “LM head”) into a vector of **logits** — one raw score per vocabulary token. For Qwen3-8B, that’s ~150,000 scores.

> **Logits are NOT probabilities.** They’re unbounded real numbers (negative, zero, or positive). A higher logit means the model considers that token more likely as the next continuation.

### Stage 3: Temperature Scaling
The logits are divided by a **temperature** parameter T:
- T < 1.0 → logits amplified → **sharper** distribution (more deterministic)
- T = 1.0 → no change
- T > 1.0 → logits dampened → **flatter** distribution (more random)

### Stage 4: Softmax → Probability Distribution
Softmax converts scaled logits into a valid probability distribution (all values in [0, 1], summing to 1.0):

```
P(token_i) = exp(logit_i / T) / sum_j( exp(logit_j / T) )
```

### Stage 5: Sampling → Selected Token
A token is drawn from the distribution. **Greedy** decoding always picks the highest-probability token. **Sampling** draws randomly (optionally with top-k or top-p filtering).

The code below demonstrates each stage with real values from the model.

In [ ]:
import torch.nn.functional as F

prompt = "The capital of France is"
messages = [{"role": "user", "content": prompt}]
text = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

# ── STEP 1: Tokenize ──
inputs = hf_tokenizer(text, return_tensors="pt").to(hf_model.device)
print("STEP 1 — Tokenize input")
print(f"  Input IDs shape: {inputs.input_ids.shape}")
print(f"  → (batch_size={inputs.input_ids.shape[0]}, sequence_length={inputs.input_ids.shape[1]})")
print()

# ── STEP 2: Forward pass → logits ──
with torch.no_grad():
    outputs = hf_model(**inputs)
logits = outputs.logits
vocab_size = logits.shape[-1]
print("STEP 2 — Forward pass → logits")
print(f"  Logits shape: {logits.shape}")
print(f"  → One score for each of {vocab_size:,} vocabulary tokens")
print()

next_logits = logits[0, -1, :]
print(f"  Next-token logits (last position): shape {next_logits.shape}")
top_vals, top_idx = torch.topk(next_logits, 5)
print("  Top 5 raw logit scores:")
for v, i in zip(top_vals, top_idx):
    print(f"    '{hf_tokenizer.decode(i)}' → logit = {v.item():.3f}")
print()

# ── STEP 3: Temperature scaling ──
temperature = 0.7
scaled = next_logits / temperature
print(f"STEP 3 — Temperature scaling (T={temperature})")
top_sc, _ = torch.topk(scaled, 5)
print(f"  Top 5 scaled logits: {[f'{v:.3f}' for v in top_sc.tolist()]}")
print("  → Lower T = sharper (more deterministic), higher T = flatter (more random)")
print()

# ── STEP 4: Softmax → probabilities ──
probs = F.softmax(scaled, dim=-1)
print("STEP 4 — Softmax → probability distribution")
print(f"  Sum of all {vocab_size:,} probabilities: {probs.sum().item():.6f}")
top_p, top_pi = torch.topk(probs, 10)
print("  Top 10 tokens by probability:")
for p, i in zip(top_p, top_pi):
    print(f"    '{hf_tokenizer.decode(i)}' → {p.item()*100:.2f}%")
print()

# ── STEP 5: Sample ──
sampled = torch.multinomial(probs, num_samples=1)
token = hf_tokenizer.decode(sampled)
print("STEP 5 — Sample from distribution")
print(f"  Sampled token: '{token}'")
print(f"  This token had probability {probs[sampled.item()].item()*100:.2f}%")

### The Key Insight: We Can Intervene at the Logit Stage

This pipeline is the foundation of **constrained generation**. The critical observation is:

> **Between Stage 2 (logits) and Stage 4 (softmax), we can modify the logits to forbid certain tokens.**

If we set a token’s logit to **−∞** (negative infinity), then after softmax its probability becomes **exactly 0**. The model physically cannot sample that token. By choosing *which* logits to mask, we control *what* the model can say — without retraining or fine-tuning.

This is the mechanism behind every constrained generation technique: regex constraints, JSON schema enforcement, grammar-guided decoding — they all work by strategically masking logits.

## How Constrained Generation Works: Logit Masking

The simplest form of constrained generation is **logit masking**:

1. Run the forward pass to get logits for the next token
2. Identify which tokens are **valid** (allowed by our constraint)
3. Set logits of all **invalid** tokens to −∞
4. Apply softmax — invalid tokens get probability ≈ 0
5. Sample — the model is forced to pick from valid tokens only

The beauty of this approach: **the model’s relative preferences among valid tokens are preserved.** If the model preferred token A over token B with a 3:1 ratio before masking, it still prefers A over B with the same 3:1 ratio after masking. We’re not overriding the model — we’re *filtering* its output space.

Let’s demonstrate this manually by forcing the model to answer only “yes” or “no”.

In [ ]:
# Manual logit masking: force the model to answer only "yes" or "no"
prompt = "Is the sky blue? Answer with a single word."
messages = [{"role": "user", "content": prompt}]
text = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

inputs = hf_tokenizer(text, return_tensors="pt").to(hf_model.device)
with torch.no_grad():
    logits = hf_model(**inputs).logits[0, -1, :]

# Find token IDs for "yes" and "no"
yes_id = hf_tokenizer.encode("yes", add_special_tokens=False)[0]
no_id = hf_tokenizer.encode("no", add_special_tokens=False)[0]
print(f"Token IDs:  'yes' = {yes_id},  'no' = {no_id}")
print()

# ── BEFORE masking ──
probs_before = F.softmax(logits, dim=-1)
top_vals, top_idx = torch.topk(probs_before, 10)
print("BEFORE masking — top 10 tokens:")
for p, i in zip(top_vals, top_idx):
    marker = " ← (allowed)" if i.item() in (yes_id, no_id) else ""
    print(f"  '{hf_tokenizer.decode(i)}' → {p.item()*100:.4f}%{marker}")
print(f"\n  P('yes') = {probs_before[yes_id].item()*100:.4f}%")
print(f"  P('no')  = {probs_before[no_id].item()*100:.4f}%")
print()

# ── APPLY MASK: set everything except yes/no to -inf ──
masked_logits = torch.full_like(logits, float('-inf'))
masked_logits[yes_id] = logits[yes_id]
masked_logits[no_id] = logits[no_id]

# ── AFTER masking ──
probs_after = F.softmax(masked_logits, dim=-1)
print("AFTER masking — only 'yes' and 'no' survive:")
print(f"  P('yes') = {probs_after[yes_id].item()*100:.2f}%")
print(f"  P('no')  = {probs_after[no_id].item()*100:.2f}%")
print(f"  Sum      = {probs_after.sum().item():.6f}")
print(f"  Tokens with P > 0: {(probs_after > 0).sum().item()}")
print()
print("→ All probability mass is redistributed to ONLY the allowed tokens.")
print("  The model's relative preference (yes vs no) is preserved.")

In [ ]:
# outlines does exactly this — but with a sophisticated state machine
# Let's verify: constrained generation with Literal["yes", "no"]
prompt = "Is the sky blue? Answer with a single word."
messages = [{"role": "user", "content": prompt}]
chat_prompt = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

result = structured_model(chat_prompt, Literal["yes", "no"])
print(f"outlines constrained result: '{result}'")
print()

print("What outlines does under the hood at EACH generation step:")
print()
print("  1. Maintains a Finite State Machine (FSM) tracking generation state")
print("  2. Queries the FSM: 'Given current state, which tokens are valid next?'")
print("  3. Builds a binary mask over the entire vocabulary (~150K tokens)")
print("  4. Sets logits of INVALID tokens to -inf (exactly like our manual mask)")
print("  5. Runs softmax + sampling on the masked logits")
print("  6. Feeds the sampled token back to the FSM to advance its state")
print("  7. Repeats until the FSM reaches an accepting state")
print()
print("For Literal['yes', 'no'], the FSM is trivial (a few states).")
print("For a full JSON schema, the FSM can have thousands of states —")
print("but the principle is identical: mask invalid tokens, sample from valid ones.")

## Grammar-Guided Decoding: From Schema to State Machine

How does `outlines` know which tokens are valid at each step? For structured output like JSON, it uses **grammar-guided decoding**:

### The Pipeline: Pydantic → JSON Schema → Finite State Machine

1. **You define a Pydantic model** (e.g., `class Person(BaseModel): name: str; age: int`)
2. **Pydantic generates a JSON Schema** — a formal specification of valid JSON structure
3. **`outlines` compiles the JSON Schema into a Finite State Machine (FSM)** — a graph of states and transitions that encodes every valid JSON string matching the schema
4. **At each generation step**, the FSM’s current state determines which tokens can legally appear next

### Example: What the FSM enforces for `{"name": "Alice", "age": 30}`

| Position | Token | FSM allows next |
|----------|-------|-----------------|
| 0 | `{` | Only `"` (object key must follow) |
| 1 | `"` | Key characters — must be a schema field name |
| 2–5 | `name"` | Only `:` (key-value separator) |
| 6 | `:` | Only `"` or space (string value) |
| 7–N | `"Alice"` | String characters until closing `"` |
| N+1 | `,` | Only `,` or `}` |
| … | `"age":30` | Digits only (int field) |
| final | `}` | End of valid object |

The FSM is computed **once** (before generation starts) and then queried at each step — so the overhead is minimal during actual token generation.

In [ ]:
import json

class PersonInfo(BaseModel):
    name: str
    age: int
    city: str

# Show the JSON Schema that drives the FSM
schema = PersonInfo.model_json_schema()
print("Pydantic model → JSON Schema:")
print(json.dumps(schema, indent=2))
print()

# Generate with the constraint
prompt = "Extract info: John Smith is 32 years old and lives in Boston."
messages = [{"role": "user", "content": prompt}]
chat_prompt = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

result = structured_model(chat_prompt, PersonInfo, max_new_tokens=256)
parsed = PersonInfo.model_validate_json(result)
print(f"Generated JSON: {result}")
print(f"Parsed: name={parsed.name}, age={parsed.age}, city={parsed.city}")
print()

# Trace what the FSM enforces at key positions
print("FSM enforcement trace:")
print("─" * 55)
print("Position  Token         Valid next tokens")
print("─" * 55)
print('  0       {             Only " (start of key)')
print('  1-N     "name"        Must match a schema field name')
print('  N+1     :             Only : (key-value separator)')
print('  N+2     "             String value must open with "')
print('  ...     John Smith"   Any string chars, then closing "')
print('  ...     ,             Comma or } (next field or close)')
print('  ...     "age":        Key + colon for next field')
print('  ...     32            Digits only (int type)')
print('  ...     ,"city":"..." Last field, then }')
print("─" * 55)
print()
print("At EVERY position, tokens violating JSON structure get -inf logits.")
print("The model literally cannot generate malformed JSON.")

### Why This Guarantees Valid Output

Grammar-guided decoding provides a **mathematical guarantee** of structural validity:

- The FSM is derived from a formal grammar (JSON Schema). It encodes *every* valid string and *only* valid strings.
- At each step, invalid tokens receive logit = −∞ → probability = 0 after softmax.
- Since sampling draws from the probability distribution, tokens with probability 0 are **never** selected.
- Therefore, the generated sequence is guaranteed to follow the grammar.

This is fundamentally different from prompt engineering (“please output JSON”) or post-hoc parsing (validate after generation). With grammar-guided decoding:

- ✅ **Impossible** to produce invalid JSON structure
- ✅ **Impossible** to have wrong field types (string where int expected, etc.)
- ✅ **Impossible** to miss required fields
- ❌ Does **not** guarantee the *content* is correct (model could output `{"age": 999}`)

The guarantee is about **form**, not **meaning**. The model’s knowledge still determines content quality.

## 1 · Prompt-Based Constraints (Soft Constraints)

The simplest approach: we describe the desired format in the prompt and rely on the model to comply. This works surprisingly well for instruction-tuned models, but offers **no guarantee** the output will match.

In [ ]:
def display_output(output):
    """Display the model's output in a formatted manner."""
    print("Model Output:")
    print("-" * 40)
    print(output)
    print("-" * 40)
    print()

prompt = """Create a product description for {product} targeted at {target_audience}.
Use a {tone} tone and keep it under {word_limit} words.
The description should include:
1. A catchy headline
2. Three key features
3. A call to action

Product Description:""".format(
    product="smart water bottle",
    target_audience="health-conscious millennials",
    tone="casual and friendly",
    word_limit="75",
)

messages = [{"role": "user", "content": prompt}]
output = generate(messages)
display_output(output)

## 2 · Rule-Based Generation with Prompt Engineering

We can add more rigid formatting instructions to the prompt. The model is asked to produce output in a specific labeled format (COMPANY / RESPONSIBILITIES / QUALIFICATIONS / EEO). This is still a *soft* constraint — the model might deviate.

In [ ]:
prompt = """Create a job posting for a {job_title} position at {company} in {location}.
The candidate should have {experience} years of experience.
Follow these rules:
1. Start with a brief company description (2 sentences)
2. List 5 key responsibilities, each starting with an action verb
3. List 5 required qualifications, each in a single sentence
4. End with a standardized equal opportunity statement

Format the output EXACTLY as follows:
COMPANY: [Company Description]

RESPONSIBILITIES:
- [Responsibility 1]
- [Responsibility 2]
- [Responsibility 3]
- [Responsibility 4]
- [Responsibility 5]

QUALIFICATIONS:
- [Qualification 1]
- [Qualification 2]
- [Qualification 3]
- [Qualification 4]
- [Qualification 5]

EEO: [Equal Opportunity Statement]""".format(
    job_title="Senior Software Engineer",
    company="TechInnovate Solutions",
    location="San Francisco, CA",
    experience="5+",
)

messages = [{"role": "user", "content": prompt}]
output = generate(messages)
display_output(output)

## 3 · Native Regex Parsing (Post-Hoc Extraction)

After generating free-form text, we use Python's `re` module to extract structured fields. This replaces LangChain's `RegexParser` — same idea, zero dependencies.

> **Limitation:** If the model doesn't follow the expected format, the regex simply won't match and you get `None`.

In [ ]:
# We reuse the output from the previous cell.
# Define the regex pattern to extract each section.
pattern = (
    r"COMPANY:\s*([\s\S]*?)\n\s*"
    r"RESPONSIBILITIES:\s*([\s\S]*?)\n\s*"
    r"QUALIFICATIONS:\s*([\s\S]*?)\n\s*"
    r"EEO:\s*([\s\S]*)"
)

match = re.search(pattern, output)

if match:
    sections = {
        "company_description": match.group(1).strip(),
        "responsibilities": match.group(2).strip(),
        "qualifications": match.group(3).strip(),
        "eeo_statement": match.group(4).strip(),
    }
    print("Parsed Output:")
    for key, value in sections.items():
        print(f"{key.upper()}:")
        print(value)
        print()
else:
    print("⚠️  Regex did not match — the model's output deviated from the expected format.")
    print("Raw output:")
    print(output)

## 4 · Structured Generation with `outlines` + Pydantic

Instead of hoping the model follows a format and parsing after the fact, **`outlines`** constrains the model *during* token generation. We define a Pydantic schema, and `outlines` masks the logits at every decoding step so the output is **guaranteed** to be valid JSON matching the schema.

This is the key insight: structured generation is not post-processing — it is a constraint on the sampling process itself.

In [ ]:
class ProductReview(BaseModel):
    product: str
    rating: int
    pros: list[str]
    cons: list[str]
    recommendation: str

prompt = """Write a product review for Smartphone X with these constraints:
- Rating: 4 out of 5 stars
- Include exactly 3 pros and 2 cons
- End with a one-sentence recommendation

Return the review as JSON matching this schema:
{"product": str, "rating": int, "pros": [str, ...], "cons": [str, ...], "recommendation": str}"""

messages = [{"role": "user", "content": prompt}]
chat_prompt = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

result = structured_model(chat_prompt, ProductReview, max_new_tokens=512)
review = ProductReview.model_validate_json(result)

print(f"Product : {review.product}")
print(f"Rating  : {review.rating}/5")
print(f"Pros    : {review.pros}")
print(f"Cons    : {review.cons}")
print(f"Rec     : {review.recommendation}")

## 5 · Constrained Classification with `Literal` Types

`outlines` can also restrict the model's output to one of a **fixed set of strings** using Python's `Literal` type. This is perfect for classification tasks — the model is physically unable to produce any token sequence outside the allowed set.

In [ ]:
# Sentiment classification — output is guaranteed to be one of the allowed labels.
prompt = "Is this review positive or negative? 'The movie was amazing and I loved every minute of it!'"
messages = [{"role": "user", "content": prompt}]
chat_prompt = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

result = structured_model(chat_prompt, Literal["Positive", "Negative"])
print(f"Sentiment: {result}")  # Guaranteed to be "Positive" or "Negative"

# Multi-class classification
prompt2 = "Classify the topic of this text: 'The Federal Reserve raised interest rates by 25 basis points.'"
messages2 = [{"role": "user", "content": prompt2}]
chat_prompt2 = hf_tokenizer.apply_chat_template(messages2, tokenize=False, add_generation_prompt=True, enable_thinking=False)

result2 = structured_model(chat_prompt2, Literal["Politics", "Economics", "Sports", "Technology", "Science"])
print(f"Topic   : {result2}")  # Guaranteed to be one of the five categories

## The Constraint Spectrum: Choice vs Schema vs Free-Form

Not all constraints are equal. They form a **spectrum** from tight (exact match) to loose (no guarantee):

1. **Choice / Regex constraint** — output must be one of a fixed set of strings (or match a regex pattern). The FSM is small and fast.
2. **Schema constraint** — output must be valid JSON conforming to a Pydantic model. The FSM can be large but handles complex nested structures.
3. **Free-form generation** — no constraint at all. You parse the output afterward and hope it’s valid.

The code below compares all three approaches on the same task.

In [ ]:
text = "The patient has a temperature of 38.5°C, with headache and fatigue for 3 days."

# ── Level 1: Choice Constraint (Literal) ──────────────────────
print("=" * 60)
print("LEVEL 1: CHOICE CONSTRAINT (Literal type)")
print("=" * 60)
prompt1 = f"Is this patient's condition urgent? {text}"
messages1 = [{"role": "user", "content": prompt1}]
cp1 = hf_tokenizer.apply_chat_template(
    messages1, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
result1 = structured_model(cp1, Literal["urgent", "not urgent", "needs monitoring"])
print(f"Result: '{result1}'")
print("Guarantee: Output is EXACTLY one of the allowed strings")
print()

# ── Level 2: Schema Constraint (Pydantic) ────────────────────
print("=" * 60)
print("LEVEL 2: SCHEMA CONSTRAINT (Pydantic model)")
print("=" * 60)

class PatientAssessment(BaseModel):
    temperature_celsius: float
    symptoms: list[str]
    duration_days: int
    urgency: str
    recommended_action: str

prompt2 = f"Assess this patient: {text}"
messages2 = [{"role": "user", "content": prompt2}]
cp2 = hf_tokenizer.apply_chat_template(
    messages2, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
result2 = structured_model(cp2, PatientAssessment, max_new_tokens=512)
parsed2 = PatientAssessment.model_validate_json(result2)
print(f"Result: {result2}")
print("Guarantee: Valid JSON with exact field names and types")
print(f"  temperature_celsius: {parsed2.temperature_celsius} (float ✓)")
print(f"  symptoms: {parsed2.symptoms} (list[str] ✓)")
print(f"  duration_days: {parsed2.duration_days} (int ✓)")
print()

# ── Level 3: Free-form (no constraint) ────────────────────────
print("=" * 60)
print("LEVEL 3: FREE-FORM (no constraint)")
print("=" * 60)
prompt3 = f"Assess this patient and return JSON: {text}"
messages3 = [{"role": "user", "content": prompt3}]
freeform_result = generate(messages3, max_new_tokens=512)
print(f"Result:\n{freeform_result[:500]}")
try:
    json.loads(freeform_result)
    print("\n✓ Happened to be valid JSON this time — but NO guarantee next time!")
except json.JSONDecodeError as e:
    print(f"\n✗ NOT valid JSON: {e}")
print("Guarantee: NONE")

### Progressive Guarantees

| Constraint Level | Mechanism | What’s Guaranteed | What’s NOT Guaranteed |
|---|---|---|---|
| **Choice / Literal** | Small FSM over fixed strings | Output is exactly one of the allowed values | Nothing about *which* value is chosen |
| **Regex pattern** | FSM from regex | Output matches the pattern | Semantic correctness |
| **JSON Schema (Pydantic)** | Large FSM from grammar | Valid JSON, correct types, required fields present | Content accuracy, value ranges |
| **Free-form** | None | Nothing | Nothing |

**Rule of thumb:** Use the *tightest* constraint that fits your task. Classification? Use `Literal`. Structured data? Use Pydantic. Creative writing? Free-form is fine.

## When Constraints Fight the Model

Constraints guarantee **format** but can degrade **content quality**. Here’s why:

When we mask out most of the vocabulary at each step, we’re forcing the model to redistribute its probability mass among a smaller set of tokens. If the model’s preferred continuation is masked out, it must pick its second (or third, or hundredth) choice. This can lead to:

- **Awkward phrasing** — the most natural word isn’t valid JSON at that position
- **Information loss** — the model can’t express nuance within rigid field types
- **Creative degradation** — poetry, humor, and style suffer in structured formats

The next cell demonstrates this tension.

In [ ]:
# Creative writing: free-form vs forced JSON structure
class RigidPoem(BaseModel):
    line1: str
    line2: str
    line3: str
    syllable_counts: list[int]
    theme: str
    mood: str

prompt_poem = "Write a beautiful haiku about a winter morning."
messages_poem = [{"role": "user", "content": prompt_poem}]
cp_poem = hf_tokenizer.apply_chat_template(
    messages_poem, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

# Constrained version
constrained_raw = structured_model(cp_poem, RigidPoem, max_new_tokens=256)
poem = RigidPoem.model_validate_json(constrained_raw)
print("CONSTRAINED (forced into JSON):")
print(f"  {poem.line1}")
print(f"  {poem.line2}")
print(f"  {poem.line3}")
print(f"  Syllables: {poem.syllable_counts}, Theme: {poem.theme}, Mood: {poem.mood}")
print()

# Free-form version
free_poem = generate(messages_poem, max_new_tokens=256)
print("FREE-FORM (natural output):")
print(f"  {free_poem[:300]}")
print()

# Factual task: structured vs free
class PreciseAnswer(BaseModel):
    answer: float
    confidence_percent: float
    reasoning: str

prompt_math = "What is the probability of rolling two sixes in a row with fair dice?"
messages_math = [{"role": "user", "content": prompt_math}]
cp_math = hf_tokenizer.apply_chat_template(
    messages_math, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

result_math = structured_model(cp_math, PreciseAnswer, max_new_tokens=256)
precise = PreciseAnswer.model_validate_json(result_math)
print("CONSTRAINED (forced precise JSON):")
print(f"  answer: {precise.answer}")
print(f"  confidence: {precise.confidence_percent}%")
print(f"  reasoning: {precise.reasoning[:150]}...")
print()

free_math = generate(messages_math, max_new_tokens=256)
print("FREE-FORM (natural explanation):")
print(f"  {free_math[:300]}")

### The Constraint Trade-Off

There is a fundamental tension between **format reliability** and **content quality**:

```
MORE constraint → MORE format reliability, potentially LESS content quality
LESS constraint → LESS format reliability, potentially MORE content quality
```

**Practical guidelines:**
- For **API responses, data extraction, classification** → use tight constraints (Pydantic, Literal). Format matters more than prose quality.
- For **creative writing, explanations, conversations** → use loose or no constraints. Natural language quality matters more than structure.
- For **hybrid tasks** (e.g., generate a report with structured metadata) → generate the creative part free-form, then extract structured fields with a second constrained call.

## Expanded Examples: Advanced Structured Generation

The examples below demonstrate increasingly complex constrained generation patterns: deeply nested objects, multi-dimensional enum classification, and information extraction from messy unstructured text.

In [ ]:
# Nested JSON objects — outlines tracks FSM state through all nesting levels

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str

class ContactInfo(BaseModel):
    email: str
    phone: str
    address: Address

class Employee(BaseModel):
    name: str
    title: str
    department: str
    contact: ContactInfo
    skills: list[str]

prompt = """Extract employee info from this text:
Jane Doe is a Senior Data Scientist in the AI Research department.
She can be reached at jane.doe@techcorp.com or (555) 123-4567.
Her office is at 42 Innovation Drive, San Jose, CA 95134.
She specializes in NLP, computer vision, and reinforcement learning."""

messages = [{"role": "user", "content": prompt}]
cp = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

result = structured_model(cp, Employee, max_new_tokens=512)
emp = Employee.model_validate_json(result)

print(f"Name       : {emp.name}")
print(f"Title      : {emp.title}")
print(f"Department : {emp.department}")
print(f"Email      : {emp.contact.email}")
print(f"Phone      : {emp.contact.phone}")
print(f"Address    : {emp.contact.address.street}, {emp.contact.address.city}, "
      f"{emp.contact.address.state} {emp.contact.address.zip_code}")
print(f"Skills     : {emp.skills}")
print()
print("Nested schema: Address → ContactInfo → Employee")
print("The FSM tracks nesting depth and field ordering through all levels.")

In [ ]:
# Multi-dimensional classification with Literal enum fields

class DocumentClassification(BaseModel):
    topic: Literal["technology", "politics", "science", "sports", "finance"]
    sentiment: Literal["positive", "negative", "neutral"]
    urgency: Literal["high", "medium", "low"]
    language_style: Literal["formal", "informal", "technical"]

texts = [
    "Breaking: The Fed just slashed interest rates — markets are soaring! Great news for investors.",
    "A new study in Nature reveals CRISPR gene editing can now target multiple genes simultaneously with unprecedented precision.",
    "The government's new tax policy is a disaster. Small businesses will suffer tremendously under these regulations.",
]

for i, txt in enumerate(texts):
    prompt = f"Classify this text:\n\n{txt}"
    messages = [{"role": "user", "content": prompt}]
    cp = hf_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    result = structured_model(cp, DocumentClassification, max_new_tokens=256)
    doc = DocumentClassification.model_validate_json(result)
    print(f"Text {i+1}: {txt[:60]}...")
    print(f"  Topic     : {doc.topic}")
    print(f"  Sentiment : {doc.sentiment}")
    print(f"  Urgency   : {doc.urgency}")
    print(f"  Style     : {doc.language_style}")
    print()

print("Every field is constrained to its Literal choices — no hallucinated categories.")
print("The JSON schema uses 'enum' constraints, which the FSM enforces per-field.")

In [ ]:
# Structured extraction from messy, informal unstructured text

class ExtractedEvent(BaseModel):
    event_name: str
    date: str
    location: str
    organizer: str
    expected_attendance: int
    topics: list[str]

unstructured = """
Hey! Just wanted to let you know about this amazing conference coming up.
It's called TechConnect 2025 and it's happening on March 15-17, 2025 at the
Moscone Center in San Francisco. The event is organized by Digital Futures Inc.
They're expecting around 5000 attendees this year. The main topics will cover
artificial intelligence, cloud computing, cybersecurity, and blockchain.
Can't wait! Let me know if you want to go together.
"""

prompt = f"Extract the event details from this message:\n{unstructured}"
messages = [{"role": "user", "content": prompt}]
cp = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

result = structured_model(cp, ExtractedEvent, max_new_tokens=512)
event = ExtractedEvent.model_validate_json(result)

print("Extracted from unstructured text:")
print(f"  Event      : {event.event_name}")
print(f"  Date       : {event.date}")
print(f"  Location   : {event.location}")
print(f"  Organizer  : {event.organizer}")
print(f"  Attendance : {event.expected_attendance:,}")
print(f"  Topics     : {event.topics}")
print()
print("From a casual email → perfectly structured data.")
print("The schema guarantees every field is present with the correct type,")
print("regardless of how messy or informal the source text is.")

## Summary

| Approach | Guarantee | Complexity | Best for |
|---|---|---|---|
| Prompt-based constraints | None (model may deviate) | Low | Quick prototyping |
| Regex post-processing | Detects non-compliance | Medium | Extracting fields from free text |
| `outlines` + Pydantic | **100% valid JSON** | Medium | Structured API responses |
| `outlines` + `Literal` | **100% valid label** | Low | Classification tasks |

`outlines` works with **any HuggingFace causal LM** — no special fine-tuning or API support needed. It manipulates the logits during decoding, so the constraint is mathematically enforced rather than merely requested.